# GRID SEARCH

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

np.random.seed(42)
jumlah_tiket = 800

# 1. FITUR (Kondisi Penerbangan)
# H-Berapa Berangkat (1 sampai 30 hari)
h_min = np.random.randint(1, 31, jumlah_tiket) 
# Hari dalam seminggu (0=Senin, 6=Minggu)
hari = np.random.randint(0, 7, jumlah_tiket) 
# Jam Penerbangan (0 sampai 23)
jam = np.random.randint(0, 24, jumlah_tiket) 

# 2. TARGET (Harga Tiket dalam Ribuan Rupiah)
# Rumus logika dasar:
# - Harga dasar 800 ribu
# - Makin mepet (h_min kecil), harga naik (+ 20rb/hari)
# - Weekend (hari 5,6), harga naik (+ 150rb)
# - Jam sibuk (jam 8-17), harga naik (+ 100rb)
harga_dasar = 800
tambahan_mepet = (30 - h_min) * 20
tambahan_weekend = np.where(hari >= 5, 150, 0)
tambahan_jam_sibuk = np.where((jam >= 8) & (jam <= 17), 100, 0)

# Total Harga + Noise (Biar AI mikir dikit)
harga_tiket = harga_dasar + tambahan_mepet + tambahan_weekend + tambahan_jam_sibuk + np.random.normal(0, 50, jumlah_tiket)

# 3. BUNGKUS JADI DATAFRAME & SPLIT
df_tiket = pd.DataFrame({'H_Min': h_min, 'Hari': hari, 'Jam': jam})
y_tiket = pd.Series(harga_tiket)

X_train, X_test, y_train, y_test = train_test_split(df_tiket, y_tiket, test_size=0.2, random_state=42)

print("✅ Data Tiket Pesawat Siap!")
print(df_tiket.head())

✅ Data Tiket Pesawat Siap!
   H_Min  Hari  Jam
0      7     3    7
1     20     0    0
2     29     1   22
3     15     1    2
4     11     3   23


In [2]:
df_tiket

,H_Min,Hari,Jam
0,7,3,7
1,20,0,0
2,29,1,22
3,15,1,2
4,11,3,23
...,...,...,...
795,29,6,19
796,11,3,17
797,11,1,13
798,18,6,1


buat grid parameter

In [68]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error

model_mentah = DecisionTreeRegressor(random_state=42)

daftar_pilihan = {
    'max_depth' : [2, 3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18],
    'min_samples_split' : [1000 ,100,5,6,7,8,9,9,10,11,12,13,14,15]
}

pencari = GridSearchCV(
    estimator = model_mentah,
    param_grid = daftar_pilihan,
    cv = 5, #setiap kombinasi diuji sebanyak 5 kali,
    scoring = 'neg_mean_squared_error',
    n_jobs = 1 #gunakan seluruh tenaga komputer
)
pencari.fit(X_train, y_train)
print("Kombinasi terbaik adalah : ", pencari.best_params_)
model_terbaik = pencari.best_estimator_
prediksi_ujian = model_terbaik.predict(X_test)
error = mean_squared_error(y_test, prediksi_ujian)

print(error)

Kombinasi terbaik adalah :  {'max_depth': 7, 'min_samples_split': 12}
3903.3292452608075


## LATIHAN 2

In [70]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV

In [71]:
# ==========================================
np.random.seed(42)
jumlah_siswa = 500

# Fitur (X)
jam_belajar = np.random.uniform(1, 10, jumlah_siswa)
jam_tidur = np.random.uniform(4, 10, jumlah_siswa)
jam_game = np.random.uniform(0, 5, jumlah_siswa)

# Target (y): Nilai Ujian (Rumus logika + Noise acak)
nilai_asli = 40 + (jam_belajar * 5) + (jam_tidur * 2) - (jam_game * 3)
noise = np.random.normal(0, 5, jumlah_siswa)

# np.clip untuk memastikan nilai tidak kurang dari 0 atau lebih dari 100
nilai_akhir = np.clip(nilai_asli + noise, 0, 100) 

X = pd.DataFrame({'Belajar': jam_belajar, 'Tidur': jam_tidur, 'Game': jam_game})
y = pd.Series(nilai_akhir)

# Bagi Data (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [90]:
model = DecisionTreeRegressor()

param = {
    'max_depth' : [2,3,4,5,6,7,8,9,10],
    'min_samples_split' : [2,3,4,5,6,7,8,9,10]
}

pencari2 = GridSearchCV(
    estimator = model_mentah,
    param_grid = param,
    cv = 5, 
    scoring = 'neg_mean_squared_error',
    n_jobs = 1
)
pencari2.fit(X_train, y_train)
print(pencari2.best_params_)

model_terbaik = pencari2.best_estimator_
model_predik = model_terbaik.predict(X_test)
model_error = mean_squared_error(y_test, model_predik)
print(f'Parameter yang diambil adalah ', pencari2.best_params_)
print(f"Nilai error ujian pada model yang telah dilakukan hyperparameter tuning adalah : {model_error:.2f}" )

{'max_depth': 6, 'min_samples_split': 10}
Parameter yang diambil adalah  {'max_depth': 6, 'min_samples_split': 10}
Nilai error ujian pada model yang telah dilakukan hyperparameter tuning adalah : 43.42


In [91]:
45.53

45.53